# Week 4 Lab: Time Series Forecasting & Recommender Systems

## Part 1: Time Series Forecasting


In [ ]:
# Run this cell once to install any missing dependencies
!pip install -q statsmodels


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

plt.rcParams['figure.figsize'] = (12, 4)
plt.rcParams['figure.dpi'] = 100
%matplotlib inline

### 1.1 Generate and Explore the Data

In [ ]:
# Generate 2 years of daily retail sales data
np.random.seed(42)
days = 365 * 2
dates = pd.date_range('2024-09-01', periods=days, freq='D')

# Hidden components (pretend you don't know these!)
trend = np.linspace(100, 170, days)
weekly_seasonality = 18 * np.sin(2 * np.pi * np.arange(days) / 7)
yearly_seasonality = 25 * np.sin(2 * np.pi * (np.arange(days) - 80) / 365.25)
holiday_bumps = np.zeros(days)
for d in [350, 351, 352, 715, 716, 717]:  # late Dec spikes in both years
    if d < days:
        holiday_bumps[d] = 40 + np.random.normal(0, 5)
noise = np.random.normal(0, 6, days)

sales = trend + weekly_seasonality + yearly_seasonality + holiday_bumps + noise
sales = np.maximum(sales, 0).round(1)

df = pd.DataFrame({'date': dates, 'sales': sales})
df = df.set_index('date')
print(f"Dataset shape: {df.shape}")
print(f"Date range: {df.index.min()} to {df.index.max()}")
df.head(10)

In [ ]:
# Plot the full time series
fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(df.index, df['sales'], linewidth=0.7)
ax.set_title('Daily Retail Sales')
ax.set_xlabel('Date')
ax.set_ylabel('Sales')
plt.tight_layout()
plt.show()

### ✏️ Exercise 1: Visual Inspection

1. Is there a **trend**? If so, describe its direction and approximate magnitude.
2. Can you spot **seasonality**? At what frequency (daily/weekly/monthly/yearly)? How can you tell?
3. Are there any **anomalies or outliers**? When do they occur and what might cause them?

### 1.2 Decomposition

Let's verify your observations by decomposing the series.

In [ ]:
from statsmodels.tsa.seasonal import seasonal_decompose

# Decompose with a 7-day period (weekly seasonality)
decomp = seasonal_decompose(df['sales'], model='additive', period=7)

fig, axes = plt.subplots(4, 1, figsize=(14, 10), sharex=True)
decomp.observed.plot(ax=axes[0], title='Observed', linewidth=0.7)
decomp.trend.plot(ax=axes[1], title='Trend', linewidth=0.7)
decomp.seasonal.plot(ax=axes[2], title='Weekly Seasonality', linewidth=0.7)
decomp.resid.plot(ax=axes[3], title='Residual', linewidth=0.7)
plt.tight_layout()
plt.show()

### 1.3 Autocorrelation

Autocorrelation tells us how correlated a value is with its past values at different lags. This directly informs which lag features to create.

In [ ]:
from statsmodels.graphics.tsaplots import plot_acf

fig, ax = plt.subplots(figsize=(12, 4))
plot_acf(df['sales'].dropna(), lags=30, ax=ax)
ax.set_title('Autocorrelation (up to 30 days)')
ax.set_xlabel('Lag (days)')
plt.tight_layout()
plt.show()

### ✏️ Exercise 2: Feature Engineering

Look at the autocorrelation plot above.

**Part A:** Which lag values show the strongest correlation? Why do those specific lags make sense given what you know about the data?

**Part B:** Based on your answer, complete the feature engineering function below. The lag features and one rolling feature are done for you. You need to add:
- At least 2 more rolling window features (different window sizes or statistics)
- All the calendar features listed in the docstring

In [ ]:
def create_features(data):
    """
    Create time series features from a sales DataFrame.
    
    Required features:
    - Lag features: lag_1, lag_7, lag_14, lag_28 (already done)
    - Rolling features: rolling_mean_7 (done), plus at least 2 more
      (e.g., rolling_std_7, rolling_mean_28, rolling_min_7, rolling_max_7)
    - Calendar features: day_of_week, month, is_weekend, day_of_year, week_of_year
    """
    df_feat = data.copy()
    
    # --- Lag features (given) ---
    df_feat['lag_1'] = df_feat['sales'].shift(1)
    df_feat['lag_7'] = df_feat['sales'].shift(7)
    df_feat['lag_14'] = df_feat['sales'].shift(14)
    df_feat['lag_28'] = df_feat['sales'].shift(28)
    
    # --- Rolling features ---
    df_feat['rolling_mean_7'] = df_feat['sales'].shift(1).rolling(window=7).mean()
    
    # ✏️ YOUR CODE: Add at least 2 more rolling features below
    # (e.g., rolling_std_7, rolling_mean_28, rolling_min_7)
    # Remember to use .shift(1) before .rolling() to avoid data leakage!
    





    
    
    # ✏️ YOUR CODE: Add calendar features below
    # day_of_week, month, is_weekend, day_of_year, week_of_year
    # Hint: df_feat.index is a DatetimeIndex -> look up its .dt accessor or attributes
    





    
    
    return df_feat

df_feat = create_features(df)
print(f"Features created: {[c for c in df_feat.columns if c != 'sales']}")
print(f"Shape before dropping NaN: {df_feat.shape}")
df_feat = df_feat.dropna()
print(f"Shape after dropping NaN:  {df_feat.shape}")
df_feat.head()

### ✏️ Exercise 3: Train/Test Split

Complete the train/test split below. Use the last 90 days as the test set and everything before as training. Remember: you must split by time, not randomly. Why would `train_test_split(X, y, test_size=0.1, random_state=42)` from sklearn be **wrong** here? What specific problem would it cause?

In [ ]:
target = 'sales'
features = [c for c in df_feat.columns if c != target]

# ✏️ YOUR CODE: split into train/test using the LAST 90 DAYS as test
# Create: X_train, X_test, y_train, y_test






print(f"Train: {X_train.shape[0]} days ({X_train.index.min().date()} to {X_train.index.max().date()})")
print(f"Test:  {X_test.shape[0]} days ({X_test.index.min().date()} to {X_test.index.max().date()})")

### 1.4 Train and Evaluate

In [ ]:
# Train a gradient boosting model
model = GradientBoostingRegressor(
    n_estimators=100,
    max_depth=3,
    learning_rate=0.1,
    random_state=42
)
model.fit(X_train, y_train)

# Predict
y_pred_train = model.predict(X_train)
y_pred_test = model.predict(X_test)

# Metrics
print(f"Train MAE: {mean_absolute_error(y_train, y_pred_train):.2f}")
print(f"Test  MAE: {mean_absolute_error(y_test, y_pred_test):.2f}")
print(f"Test  RMSE: {np.sqrt(mean_squared_error(y_test, y_pred_test)):.2f}")
print(f"\nBaseline (predict yesterday's value):")
naive_pred = df_feat.loc[X_test.index, 'lag_1']
print(f"Test  MAE: {mean_absolute_error(y_test, naive_pred):.2f}")

In [ ]:
# Plot predictions vs actual
fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(y_test.index, y_test.values, label='Actual', linewidth=1)
ax.plot(y_test.index, y_pred_test, label='Predicted', linewidth=1, alpha=0.8)
ax.set_title('Test Set: Actual vs Predicted Sales')
ax.set_xlabel('Date')
ax.set_ylabel('Sales')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Feature importance
importance = pd.Series(model.feature_importances_, index=features).sort_values(ascending=True)
fig, ax = plt.subplots(figsize=(8, 6))
importance.plot.barh(ax=ax)
ax.set_title('Feature Importance')
ax.set_xlabel('Importance')
plt.tight_layout()
plt.show()

### ✏️ Exercise 4: Interpret Your Results

Look at the prediction plot and the feature importance chart above, then answer:

1. Which are the **top 3 most important features**? Does their ranking surprise you: why or why not?
2. Does the model beat the naive baseline (predicting yesterday's value)? By how much?
3. Looking at the prediction plot: where does the model do **well**, and where does it struggle? Any pattern to the errors?

---
## Part 2: Recommender Systems


In [ ]:
# Create a small movie ratings dataset with clear patterns
movies = [
    'The Matrix', 'John Wick', 'Die Hard',           # Action
    'Inception', 'Interstellar', 'Arrival',           # Sci-Fi/Thriller
    'The Notebook', 'Pride & Prejudice', 'Titanic',   # Romance
    'Superbad', 'The Hangover', 'Bridesmaids',        # Comedy
]

users = ['Alice', 'Bob', 'Carol', 'Dave', 'Eve', 'Frank', 'Grace', 'Hank']

# Ratings (0 = not rated). Patterns:
# Alice, Bob, Carol: prefer Action/Sci-Fi
# Dave, Eve: prefer Romance/Comedy 
# Frank, Grace, Hank: mixed tastes
ratings_data = np.array([
    #  Mat  JW   DH   Inc  Int  Arr  Not  P&P  Tit  Sup  Han  Bri
    [  5,   4,   5,   5,   4,   0,   0,   0,   0,   2,   0,   0 ],  # Alice
    [  4,   5,   4,   0,   5,   4,   0,   1,   0,   0,   2,   0 ],  # Bob
    [  5,   0,   4,   4,   0,   5,   1,   0,   2,   0,   0,   0 ],  # Carol
    [  0,   1,   0,   2,   0,   0,   5,   5,   4,   4,   5,   4 ],  # Dave
    [  1,   0,   2,   0,   0,   0,   4,   5,   5,   5,   4,   0 ],  # Eve
    [  4,   3,   0,   5,   4,   3,   3,   0,   4,   2,   0,   3 ],  # Frank
    [  3,   0,   3,   0,   3,   0,   4,   4,   0,   5,   5,   4 ],  # Grace
    [  0,   4,   0,   4,   0,   3,   0,   3,   0,   4,   0,   5 ],  # Hank
], dtype=float)

# Replace 0 with NaN for 'not rated'
ratings = pd.DataFrame(ratings_data, index=users, columns=movies)
ratings_nan = ratings.replace(0, np.nan)

print("Rating matrix (NaN = not rated):")
ratings_nan

In [ ]:
# Visualize the sparsity pattern
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Heatmap of ratings
im = axes[0].imshow(ratings_nan.values, cmap='YlOrRd', aspect='auto', vmin=1, vmax=5)
axes[0].set_xticks(range(len(movies)))
axes[0].set_xticklabels(movies, rotation=45, ha='right', fontsize=9)
axes[0].set_yticks(range(len(users)))
axes[0].set_yticklabels(users)
axes[0].set_title('Ratings (white = missing)')
plt.colorbar(im, ax=axes[0], shrink=0.8)

# Sparsity
sparsity = ratings_nan.notna().astype(int)
axes[1].imshow(sparsity.values, cmap='Blues', aspect='auto', vmin=0, vmax=1)
axes[1].set_xticks(range(len(movies)))
axes[1].set_xticklabels(movies, rotation=45, ha='right', fontsize=9)
axes[1].set_yticks(range(len(users)))
axes[1].set_yticklabels(users)
axes[1].set_title(f'Sparsity pattern (filled: {sparsity.sum().sum()}/{sparsity.size} = {sparsity.sum().sum()/sparsity.size:.0%})')

plt.tight_layout()
plt.show()

### 2.1 Content-Based Filtering

In content-based filtering, we recommend items similar to what the user already liked, based on **item features** (not other users' ratings).

In [ ]:
# Define genre features for each movie
genre_features = pd.DataFrame({
    'action':    [1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    'scifi':     [1, 0, 0, 1, 1, 1, 0, 0, 0, 0, 0, 0],
    'thriller':  [1, 1, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0],
    'romance':   [0, 0, 0, 0, 0, 0, 1, 1, 1, 0, 0, 0],
    'comedy':    [0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1],
    'drama':     [0, 0, 0, 1, 1, 0, 1, 1, 1, 0, 0, 0],
}, index=movies)

print("Movie genre features:")
genre_features

### ✏️ Exercise 5: Cosine Similarity

Content-based filtering relies on measuring **similarity between items**.

Implement cosine similarity between two vectors:

$$\text{cosine\_sim}(a, b) = \frac{a \cdot b}{\|a\| \cdot \|b\|}$$

Then use it to compute the similarity between all movie pairs and answer the questions below.

In [ ]:
def cosine_similarity(a, b):
    """
    Compute cosine similarity between two numpy arrays.
    Return 0 if either vector has zero norm.
    """
    # ✏️ YOUR CODE HERE
    





    

# Compute pairwise similarity matrix
n_movies = len(movies)
sim_matrix = np.zeros((n_movies, n_movies))
for i in range(n_movies):
    for j in range(n_movies):
        sim_matrix[i, j] = cosine_similarity(
            genre_features.iloc[i].values, 
            genre_features.iloc[j].values
        )

sim_df = pd.DataFrame(sim_matrix, index=movies, columns=movies).round(2)

# Plot
fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(sim_matrix, cmap='RdYlGn', vmin=0, vmax=1)
ax.set_xticks(range(n_movies))
ax.set_xticklabels(movies, rotation=45, ha='right', fontsize=9)
ax.set_yticks(range(n_movies))
ax.set_yticklabels(movies, fontsize=9)
# Add text annotations
for i in range(n_movies):
    for j in range(n_movies):
        ax.text(j, i, f'{sim_matrix[i,j]:.2f}', ha='center', va='center', fontsize=7)
ax.set_title('Movie Similarity (Content-Based, Cosine)')
plt.colorbar(im, ax=ax, shrink=0.8)
plt.tight_layout()
plt.show()

### ✏️ Exercise 5 (continued): Interpret the Similarity Matrix

In the similarity heatmap:

1. Which pair of movies has the **highest** similarity? Does this make sense?
2. Name two movies that have **zero** similarity. Why?
3. The Matrix has non-zero similarity with both action movies AND sci-fi movies. What is its similarity score with Inception, and why is it that specific value?

In [ ]:
# Content-based recommendation function
def recommend_content_based(user_name, ratings_df, sim_matrix_df, genre_df, n=3):
    """Recommend top-n movies for a user based on content similarity."""
    user_ratings = ratings_df.loc[user_name]
    rated_mask = user_ratings.notna()
    unrated_movies = user_ratings[~rated_mask].index.tolist()
    
    scores = {}
    for movie in unrated_movies:
        # Weighted average: similarity to rated movies × those ratings
        sims = sim_matrix_df.loc[movie, rated_mask]
        rats = user_ratings[rated_mask]
        if sims.sum() > 0:
            scores[movie] = (sims * rats).sum() / sims.sum()
        else:
            scores[movie] = 0
    
    return sorted(scores.items(), key=lambda x: x[1], reverse=True)[:n]

# Test it
print("Content-based recommendations for Alice (likes Action/Sci-Fi):")
for movie, score in recommend_content_based('Alice', ratings_nan, sim_df, genre_features):
    print(f"  {movie}: predicted score = {score:.2f}")

print("\nContent-based recommendations for Dave (likes Romance/Comedy):")
for movie, score in recommend_content_based('Dave', ratings_nan, sim_df, genre_features):
    print(f"  {movie}: predicted score = {score:.2f}")

### 2.2 Collaborative Filtering

Now let's ignore item features entirely and use **only the rating patterns** across users.

### ✏️ Exercise 6: User-Based Collaborative Filtering

Complete the function below. The logic:
1. Compute similarity between the target user and all other users (using ratings they **both** rated)
2. For each unrated movie, predict the rating as a weighted average of similar users' ratings

The similarity function is provided. You need to fill in the prediction logic.

In [ ]:
def user_similarity(user_a, user_b, ratings_df):
    """
    Cosine similarity between two users based on their co-rated movies.
    """
    a = ratings_df.loc[user_a]
    b = ratings_df.loc[user_b]
    # Only use movies both users have rated
    common = a.notna() & b.notna()
    if common.sum() < 2:
        return 0.0
    return cosine_similarity(a[common].values, b[common].values)


def recommend_collaborative(target_user, ratings_df, n_neighbors=3, n_recs=3):
    """
    User-based collaborative filtering.
    
    Steps:
    1. Compute similarity between target_user and every other user
    2. Find the top n_neighbors most similar users
    3. For each movie the target hasn't rated:
       - predicted_score = weighted avg of neighbors' ratings
         (weight = similarity, only count neighbors who rated that movie)
    4. Return top n_recs movies by predicted score
    """
    all_users = ratings_df.index.tolist()
    target_ratings = ratings_df.loc[target_user]
    
    # Step 1: compute similarities
    similarities = {}
    for other_user in all_users:
        if other_user != target_user:
            similarities[other_user] = user_similarity(target_user, other_user, ratings_df)
    
    # Step 2: get top neighbors
    neighbors = sorted(similarities.items(), key=lambda x: x[1], reverse=True)[:n_neighbors]
    print(f"  Top {n_neighbors} neighbors: {[(n, f'{s:.2f}') for n, s in neighbors]}")
    
    # Step 3: predict ratings for unrated movies
    unrated = target_ratings[target_ratings.isna()].index.tolist()
    predictions = {}
    
    # ✏️ YOUR CODE: for each unrated movie, compute the weighted average
    # of the neighbors' ratings (weight = similarity score).
    # Only include neighbors who actually rated that movie.
    # If no neighbor rated it, skip it.
    






    
    
    # Step 4: return top n_recs
    return sorted(predictions.items(), key=lambda x: x[1], reverse=True)[:n_recs]


print("Collaborative filtering recommendations for Alice:")
for movie, score in recommend_collaborative('Alice', ratings_nan):
    print(f"  {movie}: predicted score = {score:.2f}")

print("\nCollaborative filtering recommendations for Dave:")
for movie, score in recommend_collaborative('Dave', ratings_nan):
    print(f"  {movie}: predicted score = {score:.2f}")

### ✏️ Exercise 7: Compare the Two Approaches

Look at the recommendations from content-based and collaborative filtering for Alice and Dave.

1. Are the recommendations the **same or different**? Pick one specific movie that appears in one method's list but not the other's, and explain why.
2. **Collaborative filtering can surprise you**: it might recommend a movie from a genre the user has never rated. Can you find an example of this in your results? Why does this happen?
3. Which method would work better for a **brand new movie** that nobody has rated yet? Why?

### 2.3 Matrix Factorization: Singular value decomposition (SVD)

Instead of explicit neighbors, we can learn **latent factors** that capture hidden patterns.

In [ ]:
# Fill missing ratings with the movie's mean for SVD
# (SVD needs a complete matrix -> this is a simple imputation strategy)
ratings_filled = ratings_nan.copy()
ratings_filled = ratings_filled.fillna(ratings_filled.mean())

# Center the ratings (subtract each movie's mean)
movie_means = ratings_filled.mean(axis=0)
ratings_centered = ratings_filled - movie_means

print("Filled & centered rating matrix:")
ratings_centered.round(2)

In [ ]:
# Perform SVD
# R ≈ U @ diag(S) @ Vt
# U: user-factor matrix, S: singular values, Vt: factor-movie matrix

U, S, Vt = np.linalg.svd(ratings_centered.values, full_matrices=False)

print(f"U shape:  {U.shape}  (users × factors)")
print(f"S shape:  {S.shape}  (singular values)")
print(f"Vt shape: {Vt.shape} (factors × movies)")
print(f"\nSingular values: {S.round(2)}")

# How much variance does each factor explain?
variance_explained = (S ** 2) / (S ** 2).sum() * 100
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(range(1, len(S) + 1), variance_explained)
ax.set_xlabel('Factor')
ax.set_ylabel('Variance Explained (%)')
ax.set_title('Singular Values: How Much Does Each Factor Matter?')
ax.set_xticks(range(1, len(S) + 1))
plt.tight_layout()
plt.show()

In [ ]:
# Keep only the top k factors
k = 3
U_k = U[:, :k]
S_k = np.diag(S[:k])
Vt_k = Vt[:k, :]

# Reconstruct the approximated ratings
ratings_approx = pd.DataFrame(
    U_k @ S_k @ Vt_k + movie_means.values,  # add means back
    index=users, 
    columns=movies
).round(2)

print("Reconstructed ratings (k=3 factors):")
ratings_approx

### Interpret the Latent Factors

The SVD has compressed each user into a 3-dimensional vector and each movie into a 3-dimensional vector. Let's see what these dimensions might mean.

In [ ]:
# Visualize user and movie embeddings in the first 2 latent dimensions
user_embeddings = U_k @ S_k  # scale by singular values
movie_embeddings = (S_k @ Vt_k).T  # movies in same space

fig, ax = plt.subplots(figsize=(10, 8))

# Plot users
for i, user in enumerate(users):
    ax.scatter(user_embeddings[i, 0], user_embeddings[i, 1], 
              s=100, c='steelblue', zorder=5)
    ax.annotate(user, (user_embeddings[i, 0], user_embeddings[i, 1]),
               fontsize=10, fontweight='bold', color='steelblue',
               textcoords='offset points', xytext=(5, 5))

# Plot movies
for i, movie in enumerate(movies):
    ax.scatter(movie_embeddings[i, 0], movie_embeddings[i, 1],
              s=80, c='tomato', marker='s', zorder=5)
    ax.annotate(movie, (movie_embeddings[i, 0], movie_embeddings[i, 1]),
               fontsize=9, color='tomato',
               textcoords='offset points', xytext=(5, -10))

ax.axhline(y=0, color='gray', linewidth=0.5)
ax.axvline(x=0, color='gray', linewidth=0.5)
ax.set_xlabel('Latent Factor 1')
ax.set_ylabel('Latent Factor 2')
ax.set_title('Users (blue) and Movies (red) in Latent Space')
ax.legend(['Users', 'Movies'], loc='upper right')
plt.tight_layout()
plt.show()

### ✏️ Exercise 8

1. **Clusters:** Do users form any groups in the latent space? Do these groups match the preference patterns in the original rating matrix? (Check back!)
2. **Proximity:** Find a user and a movie that are close together in the plot. Look up that user's actual rating for that movie (or whether it's unrated). Does proximity = liking?
3. **What do the axes mean?** The latent factors have no predefined meaning. But look at which movies are on the left vs right of Factor 1, and which are top vs bottom on Factor 2. Can you give each axis a rough label (e.g., "action vs romance")?

### ✏️ Exercise 9: SVD Recommendations

Complete the function below to use the reconstructed ratings matrix for recommendations.
This is simpler than it sounds -> the SVD has already filled in the missing ratings!

In [ ]:
def recommend_svd(user_name, ratings_original, ratings_reconstructed, n=3):
    """
    Recommend movies using SVD-reconstructed ratings.
    
    Steps:
    1. Find movies the user has NOT rated in the original matrix
    2. Look up their predicted scores in the reconstructed matrix
    3. Return the top n by predicted score
    """
    # ✏️ YOUR CODE HERE
    









print("SVD recommendations for Alice:")
for movie, score in recommend_svd('Alice', ratings_nan, ratings_approx):
    print(f"  {movie}: predicted score = {score:.2f}")

print("\nSVD recommendations for Dave:")
for movie, score in recommend_svd('Dave', ratings_nan, ratings_approx):
    print(f"  {movie}: predicted score = {score:.2f}")

print("\nSVD recommendations for Hank (mixed tastes):")
for movie, score in recommend_svd('Hank', ratings_nan, ratings_approx):
    print(f"  {movie}: predicted score = {score:.2f}")